In [1]:
from bs4 import XMLParsedAsHTMLWarning
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

import sys
sys.path.append("..")  # src 모듈을 import 하기 위해 상위 폴더를 경로에 추가

from src.edgar_client import fetch_filing_html
from src.parser import find_schedule_of_investments_tables, _row_cells, parse_filing_html

url = "https://www.sec.gov/Archives/edgar/data/1736035/000173603526000010/bxsl-20260331.htm"
html = fetch_filing_html(url)

tables = find_schedule_of_investments_tables(html)
print(f"찾은 테이블 개수: {len(tables)}")

찾은 테이블 개수: 86


In [2]:
# from src.parser import find_schedule_of_investments_tables, _table_as_of_date
# from src.edgar_client import extract_period_end_date

# tables = find_schedule_of_investments_tables(html)
# for i, table in enumerate(tables):
#     print(i, _table_as_of_date(table))

In [3]:
# from src.pipeline import build_time_series, save_to_sqlite
# from config import BXSL_CIK

# parsed, unmatched = build_time_series(BXSL_CIK, form_types=["10-Q", "10-K"], limit=8)
# print(parsed.shape, unmatched.shape)

# save_to_sqlite(parsed, unmatched)

In [4]:
# from src.edgar_client import get_recent_filings, build_document_url, fetch_filing_html
# from src.parser import find_schedule_of_investments_tables, _table_as_of_date, _row_cells
# from config import BXSL_CIK

# # 2024-06-30 filing만 다시 열어서 Aevex 관련 원본 행 확인
# filings = get_recent_filings(BXSL_CIK, form_types=["10-Q"], limit=8)
# target = [f for f in filings if f["filing_date"] == "2024-08-07"][0]
# url = build_document_url(target["cik"], target["accession_number"], target["primary_document"])
# html_2024q2 = fetch_filing_html(url)

# tables_2024q2 = find_schedule_of_investments_tables(html_2024q2)
# current_2024q2 = [t for t in tables_2024q2 if _table_as_of_date(t) == "2024-06-30"]

# for table in current_2024q2:
#     for tr in table.find_all("tr"):
#         cells = _row_cells(tr)
#         if any("Aevex" in c for c in cells):
#             print(cells)

In [5]:
# parsed[parsed["investment_name"] == "Aevex Holdings, LLC"][
#     ["investment_name", "fair_value", "period_end_date", "acquisition_date"]
# ].sort_values("period_end_date")

In [6]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import sqlite3
import pandas as pd

from src.metrics import clean_numeric_columns, detect_events
from src.normalizer import normalize_entity_name

conn = sqlite3.connect("../data/bdc_tracker.db")
positions = pd.read_sql("SELECT * FROM positions", conn)
conn.close()

positions_clean = clean_numeric_columns(positions)

In [7]:
from src.normalizer import normalize_entity_name

print(normalize_entity_name("BP Alpha Holdings, L.P."))
print(normalize_entity_name("BP Alpha Holdings, LP"))

bp alpha holdings
bp alpha holdings


In [11]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

box_events = events[events["investment_name"].str.contains("Box Co-Invest", case=False, na=False)]
print(box_events[["investment_name", "event_type", "period_end_date", "fair_value"]])

event_type
new_entry    734
exit         232
markup       208
markdown     100
Name: count, dtype: int64
                                       investment_name event_type  \
180  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
204  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
477  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
481  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
607  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   

    period_end_date  fair_value  
180      2024-09-30       155.0  
204      2024-09-30       102.0  
477      2024-12-31         0.0  
481      2024-12-31        16.0  
607      2025-03-31         0.0  


In [ ]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

box_events = events[events["investment_name"].str.contains("Box Co-Invest", case=False, na=False)]
print(box_events[["investment_name", "event_type", "period_end_date", "fair_value"]])

ModuleNotFoundError: No module named 'autoreload  # 이미 켜져있으면 생략'